# Structured output in an agent

The client writes a message in their own way. We want to extract **three specific fields** from it that can be used by Python code. We will also see where the agent stores this data after finishing its work.

In this lesson, there is one submission, one schema, and one agent call. We are not saving anything or sending it to the payment system.

In [ ]:
# All imports and minimum project path setup are in this cell.
import sys
from pathlib import Path
from typing import Literal

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from model_config import get_model
from pydantic import BaseModel

## 1. Message and Schema

`SupportCase` describes the result we expect:

- `order_id` — the order number;
- `issue` — one of two allowed categories;
- `summary` — a short summary in English.

`Literal` restricts the possible values of `issue`. Pydantic will check the types and the allowed category. It will not check whether the model correctly understood the message.

In [ ]:
SOURCE_TEXT = (
    "Good morning. Order A-174. My card was charged 79 PLN twice, "
    "even though I only bought one ticket. Please check the payment."
)

class SupportCase(BaseModel):
    order_id: str
    issue: Literal["double_charge", "other"]
    summary: str

print(SOURCE_TEXT)

## 2. The agent must return such an object

`response_format=ToolStrategy(SupportCase)` passes the schema to the agent. The agent will use the tool-calling mechanism to provide data in this format. We do not have any tools performing business operations here (`tools=[]`).

The model comes from the shared project configuration.

In [ ]:
model = get_model()
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(SupportCase),
    system_prompt=(
        "Extract the submission data. Do not add facts. "
        "Write the summary in English."
    ),
)

## 3. Agent Input and State

The input state contains `messages` with the client's message. `agent.invoke(...)` will return the **state after the run is completed**. It will contain the messages from the run as well as `structured_response`, which is an object compliant with our schema.

In [ ]:
input_state = {"messages": [{"role": "user", "content": SOURCE_TEXT}]}
print("Input state:", input_state)

result = agent.invoke(input_state)

## 4. What did the agent return?

`messages` shows the progression: the human's message and the messages related to the model's response. `structured_response` is the result that is easiest to use in Python. With `ToolStrategy`, the last message does not have to be a plain text response, so we read `structured_response` instead.

In [ ]:
print("State keys:", list(result))
print("Message types:", [message.type for message in result["messages"]])


In [ ]:
case = result["structured_response"]
case


In [ ]:
print("Data for Python:", case.model_dump())

In [ ]:
queue = "payments" if case.issue == "double_charge" else "other reports"
print("Proposed queue:", queue)

## 5. One answer, different field uses

Now we will ask the model for a different structured output:

- `text` — the response that can be seen by the client;
- `reasoning` — a short justification for the application or operator;
- `should_call_it` — a suggestion on whether the payment requires checking in another system.

`reasoning` is a **generated justification**, not a preview of the model's hidden thinking process. The flag is a decision made by the model that the code can use to determine the next step.

In [ ]:
class ReplyPlan(BaseModel):
    text: str
    reasoning: str
    should_call_it: bool


reply_agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ReplyPlan),
    system_prompt=(
        "Answer the customer in English. "
        "text: a short response for the customer, without claiming that the payment has already been checked. "
        "reasoning: one short sentence explaining why verification is needed or not. "
        "should_call_it: true, if the issue needs to be checked in the payment system."
    ),
)

reply_result = reply_agent.invoke({
    "messages": [{"role": "user", "content": SOURCE_TEXT}]
})
reply = reply_result["structured_response"]
reply

In [ ]:
print("For client:", reply.text)
print("Justification for the application:", reply.reasoning)
print("Should payment verification be proposed?", reply.should_call_it)

if reply.should_call_it:
    next_step = "Enter the report into the payment verification queue"
else:
    next_step = "Do not enter the report into the payment verification queue"
print("Simulated next step:", next_step)

The code selected the text based on `should_call_it`. It did not trigger the payment system nor did it change the application. `text` is for the client, while `reasoning` and the flag remain on the application side. The schema enforces field types; the correctness of the model's decision must be monitored separately.

[Structured output documentation](https://docs.langchain.com/oss/python/langchain/structured_output)